![](../fig/unb.jpg)
# Accuratum - Um relógio solar preciso

Este notebook serve para criar o mapa do relógio solar Accuratum.


Faculdade UnB Planaltina, Universidade de Brasília

Projeto Escola nas Estrelas


Autores: Marco Aurélio Alves Barbosa, Mikhael Ael Rocha Alves e Paulo Eduardo de Brito

Data: Abril de 2026

In [ ]:
import sys
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

import astropy
import geopy
import numpy as np
import timezonefinder

from timezonefinder import timezone_at
from accuratum.location import location_to_latitude_longitude

print('python', sys.version)
print('numpy', np.__version__)
print('astropy', astropy.__version__)
print('geopy', geopy.__version__ )
# print('timezonefinder', timezonefinder)

### User data timezone, location and current date

- **ATTENTION USER! You have to change the variable `location_str`** to set the proper latitude, longitude, and timezone; You can use google maps to search the nearest place marked as your location, or mark your own map there.
- Standard time zone is specified by the IANA time zone database, and the name must be one of those indicated by "TZ identifier" list, which can be found [on Wikipedia](https://en.wikipedia.org/wiki/List_of_tz_database_time_zones) or [here](https://github.com/jannikmi/timezonefinder/blob/master/timezonefinder/data/timezone_names.txt). Please check if it was retrieved correctly;
- the python package 'tzdata' must be installed;
    - run: `pip install tzdata` if timezones are not working.

In [ ]:
# In this example `location_str='fup planaltina'` corresponds to:
# lat, lon = -15.6006489, -47.6580608; tz_str = "America/Sao_Paulo";

# location_str = 'rodoviária do plano piloto, brasília'
location_str = 'belem, brazil'

lat, lon = location_to_latitude_longitude(location_str)
# lat = lon = 0.0
tz_str = timezone_at(lat=lat, lng=lon)
tz = ZoneInfo(tz_str)
print(f'ATTENTION: confirm that your timezone location is "{tz_str}".')

utc = ZoneInfo('UTC')

The solstice day typically is 21 but for some solstices in some years it can be 20. 
Ideally it should be calculated for each solstice on each year but we are just setting it to 21.

In [ ]:
# Finding the proper frame periods for drawing day lines
today = datetime.now(tz = tz)
this_year = today.year
last_year = this_year - 1
next_year = this_year + 1
june = 6
december = 12

default_solstice_day = 21
first_solstice_day = default_solstice_day
middle_solstice_day = default_solstice_day
last_solstice_day = default_solstice_day

first_solstice = datetime(last_year,december,first_solstice_day, tzinfo=tz)
middle_solstice = datetime(this_year,june,middle_solstice_day, tzinfo=tz)
last_solstice = datetime(this_year,december,last_solstice_day, tzinfo=tz)

frame_periods = [ [first_solstice, middle_solstice], [middle_solstice, last_solstice] ]

In [ ]:
# Calculate grids of dayline points

daylines_grids = list()
hourlines_grids = list()

for frame_period in frame_periods:

    # creating daylines grid (horizontal lines on screen)
    sunrise, sunset = 6, 18   # It is dependent on the period and location, and could be automated.

    first_date = frame_period[0]
    first_date = np.datetime64(first_date.replace(hour=sunrise+1)
                                        .astimezone(utc)
                                        .replace(tzinfo=None),
                            'h',
                            )
    last_date = frame_period[1]
    last_date = np.datetime64(last_date.replace(hour=sunrise+1)
                                    .astimezone(utc)
                                    .replace(tzinfo=None),
                            'h',
                            )
    days_step =  np.timedelta64(timedelta(days=7),'D')
    offset = np.timedelta64(timedelta(days=2),'D')
    days = np.arange(first_date, last_date + offset, days_step)

    time_range = np.timedelta64(timedelta(hours=( (sunset-1) -(sunrise+1))), 'm' )
    time_step = np.timedelta64(timedelta(minutes=1),'m')
    offset = time_step
    times = np.arange( 0, time_range + offset, time_step)

    grid_times, grid_days = np.meshgrid(times,days)
    daylines_grids.append(grid_days + grid_times)

    # creating hourlines grid (vertical lines on screen)
    time_range = np.timedelta64(timedelta(hours=( (sunset-1) -(sunrise+1))), 'm' )
    time_step = np.timedelta64(timedelta(minutes=10),'m')
    offset = np.timedelta64(timedelta(minutes=1),'s')
    times = np.arange( 0, time_range+offset, time_step)

    days_step =  np.timedelta64(timedelta(days=1),'D') # decrease to make hourlines detailed
    offset = np.timedelta64(timedelta(days=1),'D')
    days = np.arange(first_date, last_date + offset, days_step)

    grid_days, grid_times, = np.meshgrid(days,times)

    hourlines_grids.append(grid_days + grid_times)

In [ ]:
# Working with astropy to create blocks of x, y positions (daylines and hourlines)
from astropy.coordinates import EarthLocation, AltAz, get_sun
from astropy.units import deg
from astropy.time import Time

# Build a reference frame on Earth Surface
location = EarthLocation(lat=lat*deg, lon=lon*deg)
frame = AltAz(location=location, pressure=0)
plumb_length = 1.0

# for n in 0,1:   # this would go into first and second frames.
n = 0
blocks = (daylines_grids[n], hourlines_grids[n])
blocks_x = list()
blocks_y = list()
for block_n in range(len(blocks)):
    ap_lines = Time(blocks[block_n])

    # Calculating the sun position for all specified points
    sun = get_sun(ap_lines).transform_to(frame)

    sun_alt = sun.alt.value
    sun_az = sun.az.value

    # Calculating the shadow of the plumb
    shadow_length = plumb_length / np.tan(np.deg2rad(sun_alt))
    blocks_x.append( - shadow_length * np.sin(np.deg2rad(sun_az)) )
    blocks_y.append( shadow_length * np.cos(np.deg2rad(sun_az)) )


In [ ]:
# Ploting daylines only
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

fig, ax = plt.subplots(figsize=(8,6))
ax.set_aspect('equal')

for x,y in zip(blocks_x, blocks_y):
    nlines = len(x)
    for n in range(nlines):
        ax.plot(x[n],y[n],'-', color='green', linewidth=0.5)

# Institutional Logos
logo = mpimg.imread('../fig/unb_basic.jpg')
# logo_ax = fig.add_axes([0.14, 0.75, 0.12, 0.12], anchor='NW', zorder=10)
logo_ax = fig.add_axes([0.12, 0.75, 0.12, 0.12], zorder=10)
logo_ax.imshow(logo)
logo_ax.axis('off')

logo2 = mpimg.imread('../fig/escola_nas_estrelas.jpeg')
logo2_ax = fig.add_axes([0.14, 0.60, 0.12, 0.12], anchor='NW', zorder=10)
logo2_ax.imshow(logo2)
logo2_ax.axis('off')

# plumb position
xp = yp = 0
ax.scatter(xp, yp, s=10, facecolors='none', edgecolors='k', linewidths=0.5, zorder=5)

# ax.set_xlim(-13,6)
# ax.set_xlim(-10,6)
# ax.set_ylim(-2,6)
plt.show()

Useful stuff

- specifying data in cm: https://matplotlib.org/stable/gallery/subplots_axes_and_figures/figure_size_units.html
- changing image size: https://www.geeksforgeeks.org/python/how-to-change-the-size-of-figures-drawn-with-matplotlib/